# Job Queue with Status Page Example

This notebook demonstrates a job queue system with a comprehensive status page and job management features.

In [1]:
from fasthtml.common import *
from starlette.responses import StreamingResponse, JSONResponse
import uuid, time, threading
from datetime import datetime
from typing import Dict, Any

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_sizes
from cjm_fasthtml_daisyui.components.actions.modal import modal, modal_box, modal_action
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors
from cjm_fasthtml_daisyui.components.feedback.alert import alert, alert_colors
from cjm_fasthtml_daisyui.components.data_display.card import card
from cjm_fasthtml_daisyui.components.data_display.table import table, table_modifiers
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_colors
from cjm_fasthtml_daisyui.components.data_display.stat import stat, stat_title, stat_value, stats
from cjm_fasthtml_daisyui.components.data_input.text_input import text_input
from cjm_fasthtml_daisyui.components.data_input.select import select
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m, space
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size, text_color
from cjm_fasthtml_tailwind.utilities.sizing import w, max_w, max_h, container
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.utilities.borders import rounded
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, gap
from cjm_fasthtml_tailwind.utilities.effects import shadow
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app
app, rt = create_test_app(theme=DaisyUITheme.BUSINESS)

# Initialize with history for job tracking
monitor = ProgressMonitor(keep_history=True, history_limit=200)
runner = JobRunner(monitor)

# Job metadata storage
job_metadata: Dict[str, Any] = {}
job_results: Dict[str, Any] = {}

In [4]:
# Cancellable job runner extension
class CancellableJobRunner(JobRunner):
    def __init__(self, monitor):
        super().__init__(monitor)
        self._stop_events = {}
    
    def start_cancellable(self, job_id, fn, *args, patch_kwargs=None, **kwargs):
        stop_event = threading.Event()
        self._stop_events[job_id] = stop_event
        
        def wrapper():
            try:
                result = fn(stop_event, *args, **kwargs)
                job_results[job_id] = {"status": "success", "data": result}
            except Exception as e:
                job_results[job_id] = {"status": "error", "error": str(e)}
        
        return self.start(job_id, wrapper, patch_kwargs=patch_kwargs)
    
    def cancel(self, job_id):
        if job_id in self._stop_events:
            self._stop_events[job_id].set()
            job_metadata[job_id]["status"] = "cancelled"
            return True
        return False

# Use cancellable runner
runner = CancellableJobRunner(monitor)

In [5]:
# Various job types
def batch_processing_job(stop_event, batch_size=100, delay=0.01):
    from tqdm import tqdm
    import time
    
    results = []
    
    # Process in batches
    for batch in range(3):
        desc = f"Batch {batch + 1}/3"
        for i in tqdm(range(batch_size), desc=desc):
            if stop_event.is_set():
                return {"status": "cancelled", "processed": results}
            time.sleep(delay)
            results.append(f"item_{batch}_{i}")
    
    return {"status": "complete", "processed": results}

def data_export_job(stop_event, format_type="csv", records=200):
    from tqdm import tqdm
    import time
    
    # Fetch data
    for _ in tqdm(range(records // 2), desc="Fetching data"):
        if stop_event.is_set():
            return {"status": "cancelled"}
        time.sleep(0.01)
    
    # Format data
    for _ in tqdm(range(records // 4), desc=f"Formatting as {format_type}"):
        if stop_event.is_set():
            return {"status": "cancelled"}
        time.sleep(0.02)
    
    # Write to file
    for _ in tqdm(range(records // 4), desc="Writing to file"):
        if stop_event.is_set():
            return {"status": "cancelled"}
        time.sleep(0.01)
    
    return {"status": "complete", "file": f"export_{format_type}_{records}.{format_type}"}

def model_training_job(stop_event, epochs=10, batch_size=32):
    from tqdm import tqdm
    import time
    import random
    
    metrics = []
    
    for epoch in range(epochs):
        # Training
        for _ in tqdm(range(100), desc=f"Epoch {epoch+1}/{epochs} - Training"):
            if stop_event.is_set():
                return {"status": "cancelled", "metrics": metrics}
            time.sleep(0.005)
        
        # Validation
        for _ in tqdm(range(20), desc=f"Epoch {epoch+1}/{epochs} - Validation"):
            if stop_event.is_set():
                return {"status": "cancelled", "metrics": metrics}
            time.sleep(0.01)
        
        # Record metrics
        metrics.append({
            "epoch": epoch + 1,
            "loss": random.uniform(0.1, 1.0),
            "accuracy": random.uniform(0.7, 0.99)
        })
    
    return {"status": "complete", "metrics": metrics}

In [6]:
# Job queue management page
@rt("/")
def home():
    return create_test_page(
        "Job Queue Manager",
        Div(
            H1("Job Queue Management System", cls=combine_classes(font_size._3xl, font_weight.bold, m.b(6))),
            
            # Job creation panel
            Div(
                H2("Create New Job", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(
                    Select(
                        Option("Batch Processing", value="batch"),
                        Option("Data Export", value="export"),
                        Option("Model Training", value="training"),
                        id="job-type",
                        cls=combine_classes(select, w.full, m.b(3))
                    ),
                    Input(
                        id="job-name",
                        type="text",
                        placeholder="Job name (optional)",
                        cls=combine_classes(text_input, w.full, m.b(3))
                    ),
                    Div(
                        Button("Submit Job", id="submit-job", cls=combine_classes(btn, btn_colors.primary)),
                        Button("Clear All Completed", id="clear-completed", cls=combine_classes(btn, btn_colors.warning, m.l(2))),
                        cls=combine_classes(flex_display, gap(2))
                    ),
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Statistics
            Div(
                H2("Queue Statistics", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(id="stats", cls=combine_classes(stats, shadow(), w.full)),
                cls=str(m.b(6))
            ),
            
            # Job queue table
            Div(
                H2("Job Queue", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(id="job-queue", cls=str(overflow.x.auto)),
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Job details modal
            Dialog(
                Div(
                    H3("Job Details", cls=combine_classes(font_weight.bold, font_size.lg, m.b(4))),
                    Div(id="job-details-content"),
                    Div(
                        Button("Close", onclick="this.closest('dialog').close()", cls=combine_classes(btn, btn_sizes.sm)),
                        cls=str(modal_action)
                    ),
                    cls=combine_classes(modal_box, max_w._4xl)
                ),
                id="job-modal",
                cls=str(modal)
            ),
            
            cls=combine_classes(container, m.x.auto, p(8))
        ),
        Script("""
            let activeConnections = {};
            
            // Submit job
            document.getElementById('submit-job').onclick = async () => {
                const jobType = document.getElementById('job-type').value;
                const jobName = document.getElementById('job-name').value || '';
                
                const resp = await fetch('/api/jobs/create', {
                    method: 'POST',
                    headers: {'Content-Type': 'application/json'},
                    body: JSON.stringify({type: jobType, name: jobName})
                });
                
                if (resp.ok) {
                    document.getElementById('job-name').value = '';
                    refreshQueue();
                }
            };
            
            // Clear completed
            document.getElementById('clear-completed').onclick = async () => {
                await fetch('/api/jobs/clear', {method: 'POST'});
                refreshQueue();
            };
            
            // Refresh queue
            async function refreshQueue() {
                try {
                    // Get queue
                    const queueResp = await fetch('/api/jobs/queue');
                    const queueHtml = await queueResp.text();
                    document.getElementById('job-queue').innerHTML = queueHtml;
                    
                    // Get stats
                    const statsResp = await fetch('/api/jobs/stats');
                    const statsHtml = await statsResp.text();
                    document.getElementById('stats').innerHTML = statsHtml;
                } catch (err) {
                    console.error('Error refreshing queue:', err);
                }
            }
            
            // View job details
            window.viewJob = async (jobId) => {
                const resp = await fetch(`/api/jobs/${jobId}/details`);
                const html = await resp.text();
                document.getElementById('job-details-content').innerHTML = html;
                document.getElementById('job-modal').showModal();
                
                // Connect SSE for live updates
                if (activeConnections[jobId]) {
                    activeConnections[jobId].close();
                    delete activeConnections[jobId];
                }
                
                const sse = new EventSource(`/api/jobs/${jobId}/stream`);
                activeConnections[jobId] = sse;
                
                sse.onmessage = (e) => {
                    try {
                        const data = JSON.parse(e.data);
                        updateJobDetails(jobId, data);
                        
                        if (data.completed) {
                            sse.close();
                            delete activeConnections[jobId];
                            refreshQueue();
                        }
                    } catch (err) {
                        console.error('Error parsing SSE data:', err);
                    }
                };
                
                sse.onerror = (err) => {
                    console.error('SSE error for job', jobId, ':', err);
                    // Don't immediately close - SSE will auto-reconnect
                    setTimeout(() => {
                        if (sse.readyState === EventSource.CLOSED) {
                            delete activeConnections[jobId];
                        }
                    }, 1000);
                };
            };
            
            // Update job details in modal
            function updateJobDetails(jobId, data) {
                const container = document.getElementById('job-details-content');
                if (!container) return;
                
                // Update progress bars
                for (const [barId, bar] of Object.entries(data.bars || {})) {
                    const progressEl = container.querySelector(`#detail-progress-${barId}`);
                    const textEl = container.querySelector(`#detail-text-${barId}`);
                    
                    if (progressEl) {
                        progressEl.value = bar.pct || 0;
                        if (textEl) {
                            textEl.textContent = `${bar.desc}: ${(bar.pct || 0).toFixed(1)}% (${bar.cur || 0}/${bar.tot || 0})`;
                        }
                    }
                }
                
                // Update overall
                const overallEl = container.querySelector('#detail-overall');
                if (overallEl) {
                    overallEl.value = data.progress || 0;
                }
            }
            
            // Cancel job
            window.cancelJob = async (jobId) => {
                const resp = await fetch(`/api/jobs/${jobId}/cancel`, {method: 'POST'});
                if (resp.ok) {
                    if (activeConnections[jobId]) {
                        activeConnections[jobId].close();
                        delete activeConnections[jobId];
                    }
                    refreshQueue();
                    document.getElementById('job-modal').close();
                }
            };
            
            // Auto-refresh
            refreshQueue();
            setInterval(refreshQueue, 2000);
        """)
    )

In [7]:
# API endpoints
@rt("/api/jobs/create", methods=["POST"])
async def create_job(request):
    data = await request.json()
    job_type = data.get('type', 'batch')
    job_name = data.get('name', '')
    
    job_id = str(uuid.uuid4())
    
    # Select job function with appropriate parameters
    job_configs = {
        'batch': (batch_processing_job, {'batch_size': 50, 'delay': 0.005}),
        'export': (data_export_job, {'format_type': 'csv', 'records': 100}),
        'training': (model_training_job, {'epochs': 3, 'batch_size': 32})
    }
    
    job_fn, kwargs = job_configs.get(job_type, (batch_processing_job, {}))
    
    # Store metadata
    job_metadata[job_id] = {
        'id': job_id,
        'name': job_name or f"{job_type.title()} Job",
        'type': job_type,
        'status': 'running',
        'created_at': datetime.now().isoformat(),
        'params': kwargs
    }
    
    # Start job with appropriate throttling
    runner.start_cancellable(
        job_id,
        job_fn,
        **kwargs,
        patch_kwargs={
            "min_delta_pct": 5,
            "min_update_interval": 0.05,
            "emit_initial": True
        }
    )
    
    return JSONResponse({"job_id": job_id})

@rt("/api/jobs/queue")
def get_queue():
    jobs = monitor.all()
    
    if not jobs:
        return P("No jobs in queue", cls=str(text_color.gray(500)))
    
    rows = []
    for job_id, job in jobs.items():
        meta = job_metadata.get(job_id, {})
        
        rows.append(
            Tr(
                Td(job_id[:8] + "..."),
                Td(meta.get('name', 'Unknown')),
                Td(meta.get('type', 'unknown').title()),
                Td(
                    Progress(
                        value=str(job['overall_progress']),
                        max="100",
                        cls=combine_classes(progress, progress_colors.primary, w(20))
                    )
                ),
                Td(
                    Span(
                        "Complete" if job['completed'] else "Running",
                        cls=combine_classes(badge, badge_colors.success) if job['completed'] else combine_classes(badge, badge_colors.info)
                    )
                ),
                Td(
                    Button("View", onclick=f"viewJob('{job_id}')", cls=combine_classes(btn, btn_sizes.xs, btn_colors.primary)),
                    Button("Cancel", onclick=f"cancelJob('{job_id}')", cls=combine_classes(btn, btn_sizes.xs, btn_colors.error, m.l(1))) if not job['completed'] else ""
                )
            )
        )
    
    return Table(
        Thead(
            Tr(
                Th("ID"),
                Th("Name"),
                Th("Type"),
                Th("Progress"),
                Th("Status"),
                Th("Actions")
            )
        ),
        Tbody(*rows),
        cls=combine_classes(table, table_modifiers.zebra, w.full)
    )

@rt("/api/jobs/stats")
def get_stats():
    jobs = monitor.all()
    
    total = len(jobs)
    running = sum(1 for j in jobs.values() if not j['completed'])
    completed = total - running
    
    return Div(
        Div(
            Div("Total Jobs", cls=str(stat_title)),
            Div(str(total), cls=str(stat_value)),
            cls=str(stat)
        ),
        Div(
            Div("Running", cls=str(stat_title)),
            Div(str(running), cls=combine_classes(stat_value, text_dui.primary)),
            cls=str(stat)
        ),
        Div(
            Div("Completed", cls=str(stat_title)),
            Div(str(completed), cls=combine_classes(stat_value, text_dui.success)),
            cls=str(stat)
        ),
        cls=combine_classes(stats, shadow(), w.full)
    )

@rt("/api/jobs/{job_id}/details")
def get_job_details(job_id: str):
    import json
    snapshot = monitor.snapshot(job_id)
    meta = job_metadata.get(job_id, {})
    result = job_results.get(job_id)
    
    if not snapshot:
        return Div("Job not found", cls=combine_classes(alert, alert_colors.error))
    
    # Build progress bars
    bars = []
    for bar_id, bar in snapshot['bars'].items():
        bars.append(
            Div(
                P(f"{bar.description}: {bar.progress:.1f}% ({bar.current}/{bar.total})", id=f"detail-text-{bar_id}", cls=str(font_size.sm)),
                Progress(value=str(bar.progress), max="100", id=f"detail-progress-{bar_id}", cls=combine_classes(progress, progress_colors.accent, w.full)),
                cls=str(m.b(3))
            )
        )
    
    return Div(
        # Job info
        Div(
            P(f"ID: {job_id}", cls=combine_classes(font_size.xs, text_color.gray(500))),
            P(f"Name: {meta.get('name', 'Unknown')}", cls=str(font_weight.semibold)),
            P(f"Type: {meta.get('type', 'unknown').title()}"),
            P(f"Created: {meta.get('created_at', 'Unknown')}", cls=str(font_size.sm)),
            cls=str(m.b(4))
        ),
        
        # Overall progress
        Div(
            P(f"Overall Progress: {snapshot['overall_progress']:.1f}%", cls=combine_classes(font_weight.bold, m.b(2))),
            Progress(value=str(snapshot['overall_progress']), id="detail-overall", max="100", cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))),
        ),
        
        # Individual bars
        Div(
            H4("Task Progress:", cls=combine_classes(font_weight.semibold, m.b(2))),
            *bars
        ) if bars else "",
        
        # Results if available
        Div(
            H4("Results:", cls=combine_classes(font_weight.semibold, m.b(2))),
            Pre(json.dumps(result, indent=2), cls=combine_classes(bg_dui.base_300, p(3), rounded(), font_size.xs, overflow.auto, max_h(40))),
            cls=str(m.t(4))
        ) if result else "",
        
        # History if available
        Div(
            H4("History:", cls=combine_classes(font_weight.semibold, m.b(2))),
            P(f"{len(snapshot.get('history', []))} updates recorded", cls=str(font_size.sm)),
            cls=str(m.t(4))
        ) if snapshot.get('history') else ""
    )

@rt("/api/jobs/{job_id}/cancel", methods=["POST"])
def cancel_job(job_id: str):
    success = runner.cancel(job_id)
    return JSONResponse({"success": success})

@rt("/api/jobs/{job_id}/stream")
def stream_job(job_id: str):
    gen = sse_stream(
        monitor,
        job_id,
        interval=0.05,  # Faster updates for better responsiveness
        heartbeat=15.0,
        wait_for_start=True,
        start_timeout=10.0
    )
    return StreamingResponse(gen, media_type="text/event-stream")

@rt("/api/jobs/clear", methods=["POST"])
def clear_completed():
    monitor.clear_completed(older_than_seconds=0)
    
    # Clean up metadata and results
    for job_id in list(job_metadata.keys()):
        if not monitor.snapshot(job_id):
            del job_metadata[job_id]
            if job_id in job_results:
                del job_results[job_id]
    
    return JSONResponse({"status": "cleared"})

In [8]:
# Start server
server = start_test_server(app)
display(HTMX())

In [9]:
# Stop server when done
server.stop()